# Kumo GNN on the Netflix 5k slice — `kumo-engine`

This is `demo_netflix_gnn.ipynb` rewritten against **`kumo-engine`**, the new
local-only package (`packages/kumo-engine`) that replaces the cloud SDK's
`kumoai.LocalBackend`. Same dataset, same predictive query, same GNN pipeline --
but no `Backend`/`set_backend`, no `project_id`, and connectors/identities are
threaded through as plain Python objects instead of going through a registry.

See `docs/kumo-engine-sdk-split-design.md` for the design this notebook
exercises end to end.

## 1. Install

In [ ]:
import importlib.metadata as md

import kumo.engine as kumo

print('kumo.engine:', kumo.__version__)
for dist in ('kumo-core', 'torch'):
    try:
        print(f'{dist}:', md.version(dist))
    except md.PackageNotFoundError:
        print(f'{dist}: not installed as a distribution')

## 2. Connect

Two ways to build a `SnowflakeConnector` + its `SnowflakeIdentity`, chosen
by whether this notebook is itself running inside Snowpark Container
Services -- checked directly via `os.path.exists(SPCS_TOKEN_PATH)`, the
same OAuth-token-mount check `SnowflakeConnector.from_spcs_env()` itself
makes internally (and raises `ValueError` on, outside SPCS):

* In SPCS: `SnowflakeConnector.from_spcs_env()` builds straight from the
  environment SPCS already provides (the mounted OAuth token, `SNOWFLAKE_
  ACCOUNT`/`HOST`).
* Otherwise: `SnowflakeConnector.from_connection_toml(...)`, from a
  `[connections.<name>]` entry in Snowflake's own `connections.toml`, the
  file most people already have working before they reach Kumo.

Both mirror the cloud SDK's own methods of the same names, minus the parts
that only exist to round-trip a server-side identity registry. A key pair
is preferred over a password where a `connections.toml` entry has both.

In [ ]:
import os

from kumo_snowflake.spcs import SPCS_TOKEN_PATH

#: Both connection paths below still need an explicit target -- SPCS's
#: own SNOWFLAKE_WAREHOUSE/DATABASE/SCHEMA (if set at all) may not be this
#: demo's, and a connector addresses one exact target.
TARGET = dict(warehouse='WH_S', database='KUMO', schema_name='NETFLIX_5K')

if os.path.exists(SPCS_TOKEN_PATH):
    connector = kumo.SnowflakeConnector.from_spcs_env(**TARGET)
    print('running in Snowpark Container Services')
else:
    #: A [connections.<name>] entry in ~/.snowflake/connections.toml (or
    #: wherever SNOWFLAKE_HOME points) -- only the name appears here; the
    #: key or password itself is read straight into a SnowflakeIdentity and
    #: never printed.
    CONNECTION_NAME = 'fub'
    connector = kumo.SnowflakeConnector.from_connection_toml(
        connection_name=CONNECTION_NAME, **TARGET)
    print('not running in SPCS; using connections.toml connection',
          repr(CONNECTION_NAME))

print('connector id:', connector.id)
print('identity slug:', connector.identity.slug)

## 3. Read the catalog

`connector['table_name']` (`Connector.__getitem__`, same as the cloud SDK's
own `connector[table_name]`) returns a `SourceTable` -- a reference to that
table, with its schema resolved lazily (and cached) from the actual source
data on first access via `.column_dict`/`.columns`.

In [ ]:
print('tables in KUMO.NETFLIX_5K:', connector.table_names())

for table_name in ('Movies', 'Users', 'Ratings'):
    for col in connector[table_name].columns:
        print(f'{table_name}.{col.name}', col.dtype,
              col.stype or '- (needs column statistics)')

## 4. Define the tables

`Table.from_source_table(connector[table_name], ...)` resolves dtype/stype
the same way the cloud SDK's own `Table.from_source_table(source_table,
...)` does: cheap dtype-implied stypes first (`SourceTable.column_dict`),
then a sampled scan for anything still ambiguous. `infer_metadata()` takes
no arguments -- it re-resolves using whatever column selection/primary
key/time columns the table already carries, so it chains after
`from_source_table` (e.g. to pick up fresh source stats) and before
`with_column(...)`, the immutable equivalent of the cloud SDK's
`table[name].stype = ...` -- it returns a new `Table` rather than mutating.

In [ ]:
from kumoapi.typing import Dtype, Stype

movies = kumo.Table.from_source_table(
    connector['Movies'],
    ['MovieID', 'YearOfRelease'],  # 'Title' needs a text encoder
    primary_key='MovieID',
)
users = kumo.Table.from_source_table(connector['Users'], primary_key='userID')

# infer_metadata() takes no arguments -- unlike from_source_table, it
# reuses whatever column selection/primary key/time columns the table
# already carries (here, just time_column='date' from from_source_table),
# so it chains cleanly with with_column() for post-hoc corrections.
ratings = (
    kumo.Table.from_source_table(connector['Ratings'], time_column='date')
    .infer_metadata()
    # Foreign keys are a graph-level fact -- see the edges in section 5 --
    # so inference, which sees one table at a time, judges them on their
    # values and calls them numerical. They are keys.
    .with_column('movieID', stype=Stype.ID)
    .with_column('userID', stype=Stype.ID)
    # Stored as TEXT in Snowflake, so the catalog reports a string. A time
    # column has to be a real date, and this declaration is what makes
    # staging cast it.
    .with_column('date', dtype=Dtype.date)
)

for label, table in [('movies', movies), ('users', users),
                     ('ratings', ratings)]:
    resolved = ', '.join(f'{c.name}:{c.stype}' for c in table.definition.cols)
    print(f'{label:<8} {resolved}')

## 5. Assemble the graph

Edges are declared in the SDK's direction -- the table holding the foreign key
points at the table holding the primary key. `Graph.validate()` runs the same
structural check the cloud SDK's local backend does (portable-graph conversion
+ `kumo_core.pquery.normalize_graph`), needing only table *schemas* -- no
materialized snapshot yet.

In [ ]:
from IPython.display import display

from kumo.engine import Edge

graph = kumo.Graph(
    tables=[movies, users, ratings],
    edges=[
        Edge(src_table='Ratings', fkey='movieID', dst_table='Movies'),
        Edge(src_table='Ratings', fkey='userID', dst_table='Users'),
    ],
)
graph.validate()
display(graph.visualize())
print('graph valid. snapshot_id:', graph.id)

## 6. The predictive query

Unchanged from the original demo. `PredictiveQuery(graph=graph, query=QUERY)`
binds the graph at construction -- matching the cloud SDK's own constructor
shape -- and parses lazily (cached on first access to `validate()`/
`get_task_type()`, shared with `graph.native_graph`).

In [ ]:
QUERY = """
PREDICT COUNT_DISTINCT(Ratings.rating WHERE Ratings.rating < 3, 0, 60, days) > 0
FOR EACH Movies.MovieID
WHERE AVG(Ratings.rating, -60, 0, days) > 3
""".strip()
print(QUERY)

pquery = kumo.PredictiveQuery(graph=graph, query=QUERY)
pquery.validate()
print('\ntask type:', pquery.get_task_type())

## 7. Snapshot and generate a training table

`Graph.snapshot()` is the explicit materialization step: reads every table
through its connector, runs the result through `kumo_core` to build the
on-disk DiskGraph index. `PredictiveQuery.generate_training_table(snapshot)`
compiles the query into an actual train table -- useful to inspect/export,
though `Trainer.fit` re-derives the table itself rather than consuming this
one (matching `kumo_core.training.workflow.train_prepared_graph`'s own
materialize-then-train flow).

In [ ]:
import pandas as pd

snapshot = graph.snapshot()
print('index_path:', snapshot.index_path)

train_table = pquery.generate_training_table(snapshot)
print('train_table_path:', train_table.train_table_path)
pd.read_parquet(train_table.train_table_path).head()

## 8. Train

`Trainer.fit(train_table)` runs the same three steps the cloud SDK's `fit`
does, in this process: materialize the training task against the snapshot,
run AutoML/GNN training, and (if `export_config` is given) publish the
trained model -- the local replacement for the cloud SDK's standalone
`export_model()`.

In [ ]:
from kumoapi.model_plan import ModelPlan

trainer = kumo.Trainer(model_plan=ModelPlan())
result = trainer.fit(train_table)
print('metrics:', result.metrics)
print('\nbundle_dir:', result.bundle_dir)

The metrics above are meaningful against the real `NETFLIX_5K` table -- a
well-trained GNN should land around 0.85 test AUROC on this task.

## 9. Predict

`PredictiveQuery.generate_prediction_table(snapshot, prediction_time=...)` is
the prediction counterpart of `generate_training_table` -- compiles the query
into a materialized `PredictionTable` for inspection/export. `Trainer.predict`
re-derives its own compiled table from `(snapshot, query, prediction_time)`
rather than consuming `PredictionTable.table_path`, the same way `Trainer.fit`
re-derives its own training table rather than consuming `TrainingTable.
train_table_path`.

`Trainer.predict` reuses the just-trained model (`Trainer` carries
`_training_result` after `fit`, the same way the cloud SDK's `Trainer` carries
`_training_job_id`) rather than naming a job id. `prediction_time` is the
local replacement for the cloud SDK's `Trainer.predict(prediction_time=...)`:
there is no `connector.sql(...)` yet to compute `MAX(date)` directly, so a
known in-range date stands in for "the last rating in the data".

In [ ]:
from datetime import datetime, timezone

from kumoapi.jobs import PredictionArtifactType, WriteMode

prediction_table = pquery.generate_prediction_table(
    snapshot, prediction_time=datetime(2005, 6, 1, tzinfo=timezone.utc))
print('table_path:', prediction_table.table_path)

output_config = kumo.PredictionOutputConfig(
    connector=connector,
    artifact_type=PredictionArtifactType.PREDICTIONS,
    table_name='NETFLIX_5K_PREDICTIONS_OUT',
    write_mode=WriteMode.OVERWRITE,
)

pred_result = trainer.predict(
    prediction_table,
    output_config=output_config,
    prediction_time=datetime(2005, 6, 1, tzinfo=timezone.utc),
)
print('rows:', len(pred_result.frame))
pred_result.frame.head()

In [ ]:
# The scores are an ordinary frame, so ranking is ordinary pandas.
top = pred_result.frame.sort_values('True_PROB', ascending=False).head(5)
print('Most likely to receive a low rating in the next 60 days:')
print(top[['ENTITY', 'True_PROB']].to_string(index=False))

## 10. What is still partial

* `PredictiveQuery.suggest_model_plan` isn't implemented -- matches the
  reference cloud LocalBackend's own limitation (`kumo_core` can only
  resolve a plan after ingest; pass model plan overrides to `Trainer`
  directly instead).
* `Trainer.load` (reload a previously published model) isn't implemented yet.
* `connector.sql(...)` (arbitrary read-only SQL) isn't ported -- used above
  only to explain why `prediction_time` is hardcoded rather than computed.
* Hosted service administration and Custom SQL tables are
  out of scope for this package -- see
  `docs/kumo-engine-sdk-split-design.md`.